In [ ]:
# DETERMINISM SETUP
import os
import sys

# Set deterministic environment variables BEFORE any torch import
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Track if torch was already preloaded 
_TORCH_WAS_PRELOADED = "torch" in sys.modules
_CUBLAS_ENV_WAS_PRESET = "CUBLAS_WORKSPACE_CONFIG" in os.environ

import gc
import json
import random
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer


# ===================== REPRODUCIBILITY / SEEDING =====================
SEED = int(os.environ.get("REPRODUCIBLE_SEED", 42))
STRICT_DETERMINISM = os.environ.get("STRICT_DETERMINISM", "1").strip().lower() not in {"0", "false", "no"}
STRICT_DETERMINISM_ENABLED = False


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def _disable_strict_determinism(reason):
    global STRICT_DETERMINISM, STRICT_DETERMINISM_ENABLED
    STRICT_DETERMINISM = False
    STRICT_DETERMINISM_ENABLED = False
    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass
    print("Warning: strict deterministic algorithms disabled.")
    print("         Reason:", reason)


def configure_determinism():
    global STRICT_DETERMINISM_ENABLED
    if not STRICT_DETERMINISM:
        print("Torch deterministic algorithms: DISABLED (STRICT_DETERMINISM=0)")
        return

    if torch.cuda.is_available() and _TORCH_WAS_PRELOADED and not _CUBLAS_ENV_WAS_PRESET:
        _disable_strict_determinism(
            "torch was already imported before CUBLAS_WORKSPACE_CONFIG was set."
        )
        print("         For strict CUDA determinism in Kaggle/Colab, set")
        print("         %env CUBLAS_WORKSPACE_CONFIG=:4096:8")
        print("         and restart the kernel before running this script.")
        return

    cublas_cfg = os.environ.get("CUBLAS_WORKSPACE_CONFIG", "")
    valid_cfg = {":4096:8", ":16:8"}
    if torch.cuda.is_available() and cublas_cfg not in valid_cfg:
        _disable_strict_determinism(
            f"unsupported CUBLAS_WORKSPACE_CONFIG='{cublas_cfg}'. Use :4096:8 or :16:8."
        )
        return

    try:
        torch.use_deterministic_algorithms(True)
        STRICT_DETERMINISM_ENABLED = True
        print("Torch deterministic algorithms: ENABLED")
    except Exception as e:
        _disable_strict_determinism(str(e))


set_seed(SEED)
configure_determinism()


# ===================== CONFIGURATION =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Main fair-comparison model.
# - intfloat/multilingual-e5-base
# - sentence-transformers/paraphrase-multilingual-mpnet-base-v2
MODEL_NAME = os.environ.get("MODEL_NAME", "intfloat/multilingual-e5-base")
MODEL_PREFIX = os.environ.get("MODEL_PREFIX", "passage: " if "e5" in MODEL_NAME.lower() else "")
TRUST_REMOTE_CODE = os.environ.get("TRUST_REMOTE_CODE", "0").strip().lower() in {"1", "true", "yes"}

EN_DATA_DIR = Path(os.environ.get("EN_DATA_DIR", "/kaggle/input/datasets/anisoaraana/similarity"))
RO_DATA_DIR = Path(os.environ.get("RO_DATA_DIR", "/kaggle/input/datasets/anisoaraana/romanian-narrative-similarity-dataset"))
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PREFIX = os.environ.get("RUN_PREFIX", "multilingual_g2_en_ro_comparison")
LANGUAGES = [x.strip().lower() for x in os.environ.get("LANGUAGES", "en,ro").split(",") if x.strip()]
RO_SUFFIX = os.environ.get("RO_SUFFIX", "_ro")
TRANSLATION_CACHE_PATH = Path(os.environ.get("TRANSLATION_CACHE_PATH", RO_DATA_DIR / "translation_cache_en_ro.json"))

MAX_LEN = int(os.environ.get("MAX_LEN", 512))
STAGE1_EPOCHS = int(os.environ.get("STAGE1_EPOCHS", 1))
STAGE2_EPOCHS = int(os.environ.get("STAGE2_EPOCHS", 2))
STAGE1_LR = float(os.environ.get("STAGE1_LR", 1e-5))
STAGE2_LR = float(os.environ.get("STAGE2_LR", 6e-6))
STAGE1_BS = int(os.environ.get("STAGE1_BS", 2))
STAGE2_BS = int(os.environ.get("STAGE2_BS", 4))
PRED_BS = int(os.environ.get("PRED_BS", 16))
ENC_BS = int(os.environ.get("ENC_BS", 8))
MARGIN = float(os.environ.get("MARGIN", 0.3))

USE_SYNTHETIC_STAGE1 = os.environ.get("USE_SYNTHETIC_STAGE1", "1").strip().lower() not in {"0", "false", "no"}
USE_EXTRA_SYNTHETIC = os.environ.get("USE_EXTRA_SYNTHETIC", "1").strip().lower() not in {"0", "false", "no"}
RUN_FINAL_TEST = os.environ.get("RUN_FINAL_TEST", "1").strip().lower() not in {"0", "false", "no"}
EVAL_EACH_EPOCH = os.environ.get("EVAL_EACH_EPOCH", "1").strip().lower() in {"1", "true", "yes"}

HEAD_KEYS = [h.strip() for h in os.environ.get("HEAD_KEYS", "head1,head2").split(",") if h.strip()]
PROJ_DIM = int(os.environ.get("PROJ_DIM", 256))
HEAD_TRIPLET_WEIGHT = float(os.environ.get("HEAD_TRIPLET_WEIGHT", 0.30))
GLOBAL_TRIPLET_WEIGHT = float(os.environ.get("GLOBAL_TRIPLET_WEIGHT", 0.70))
GLOBAL_RANK_WEIGHT = float(os.environ.get("GLOBAL_RANK_WEIGHT", 0.30))

DL_NUM_WORKERS = int(os.environ.get("DL_NUM_WORKERS", 0))
USE_AMP = DEVICE == "cuda" and os.environ.get("USE_AMP", "1").strip().lower() not in {"0", "false", "no"}
AMP_DTYPE_NAME = os.environ.get("AMP_DTYPE", "float16").strip().lower()
AMP_DTYPE = torch.bfloat16 if AMP_DTYPE_NAME in {"bf16", "bfloat16"} else torch.float16

COMPARISON_JSON = OUT_DIR / f"{RUN_PREFIX}_metrics.json"
COMPARISON_MD = OUT_DIR / f"{RUN_PREFIX}_summary.md"


def _worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    os.environ["PYTHONHASHSEED"] = str(worker_seed)


# ===================== HELPERS =====================
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def norm_text(text):
    return " ".join(str(text).split())


def load_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def l2_normalize_np(x):
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)


def macro_f1_from_binary(y_true, y_pred):
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    tp = int((y_pred & y_true).sum())
    tn = int((~y_pred & ~y_true).sum())
    fp = int((y_pred & ~y_true).sum())
    fn = int((~y_pred & y_true).sum())
    f1_pos = 2 * tp / (2 * tp + fp + fn + 1e-9)
    f1_neg = 2 * tn / (2 * tn + fn + fp + 1e-9)
    return (f1_pos + f1_neg) / 2.0, tp, tn, fp, fn


def autocast_context():
    if not USE_AMP:
        return nullcontext()
    return torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE)


def make_grad_scaler():
    if not USE_AMP:
        return None
    return torch.amp.GradScaler("cuda")


def backward_step(loss, optimizer, scaler, model):
    optimizer.zero_grad(set_to_none=True)
    if scaler is None:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        return
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()


def load_translation_cache(path):
    if not path.exists():
        print(f"Warning: translation cache not found: {path}")
        return {}
    with open(path, encoding="utf-8") as handle:
        raw = json.load(handle)
    if not isinstance(raw, dict):
        raise ValueError(f"Translation cache must be a JSON object: {path}")
    return {norm_text(k): norm_text(v) for k, v in raw.items()}


TRANSLATION_CACHE = load_translation_cache(TRANSLATION_CACHE_PATH)
print(f"Translation cache entries: {len(TRANSLATION_CACHE)}")


def translate_ref_text_for_eval(english_text):
    return TRANSLATION_CACHE.get(norm_text(english_text))


def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return paths[0]


def language_paths(language):
    if language == "en":
        data_dir = EN_DATA_DIR
        return {
            "language": "English",
            "data_dir": str(data_dir),
            "synth": data_dir / "synthetic_data_for_classification.jsonl",
            "extra": data_dir / "synthetic_data_new.jsonl",
            "dev_track_a": data_dir / "dev_track_a.jsonl",
            "test_track_a": data_dir / "test_track_a.jsonl",
            "test_track_b": data_dir / "test_track_b.jsonl",
            "test_track_a_labels": data_dir / "test_track_a_labels.jsonl",
            "test_track_b_labels": data_dir / "test_track_b_labels.jsonl",
        }

    if language == "ro":
        data_dir = RO_DATA_DIR

        def ro_file(stem):
            return first_existing([
                data_dir / f"{stem}{RO_SUFFIX}.jsonl",
                data_dir / f"{stem}.jsonl",
            ])

        return {
            "language": "Romanian machine translation",
            "data_dir": str(data_dir),
            "synth": ro_file("synthetic_data_for_classification"),
            "extra": ro_file("synthetic_data_new"),
            "dev_track_a": ro_file("dev_track_a"),
            "test_track_a": ro_file("test_track_a"),
            "test_track_b": ro_file("test_track_b"),
            "test_track_a_labels": EN_DATA_DIR / "test_track_a_labels.jsonl",
            "test_track_b_labels": EN_DATA_DIR / "test_track_b_labels.jsonl",
        }

    raise ValueError(f"Unsupported language: {language}. Use en or ro.")


# ===================== MODEL =====================
def mean_pool(model_output, attention_mask):
    token_emb = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    summed = (token_emb * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return F.normalize(summed / counts, p=2, dim=1)


class MultilingualG2Model(nn.Module):
    def __init__(self, encoder, hidden_dim, proj_dim, head_keys):
        super().__init__()
        self.encoder = encoder
        self.heads = nn.ModuleDict({
            key: nn.Linear(hidden_dim, proj_dim)
            for key in head_keys
        })

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return mean_pool(out, attention_mask)

    def forward(self, input_ids, attention_mask):
        emb = self.encode(input_ids, attention_mask)
        out = {"global": emb}
        for key, head in self.heads.items():
            out[key] = F.normalize(head(emb), p=2, dim=1)
        return out


# ===================== DATASETS =====================
class SyntheticTripletDataset(Dataset):
    def __init__(self, path):
        self.data = []
        if not Path(path).exists():
            print(f"Warning: missing synthetic path, skipped: {path}")
            return
        with open(path, encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                obj = json.loads(line)
                anchor = obj.get("anchor_text", "")
                text_a = obj.get("text_a", "")
                text_b = obj.get("text_b", "")
                closer = obj.get("text_a_is_closer")
                if not anchor or not text_a or not text_b:
                    continue
                if len(anchor) < 20 or len(text_a) < 20 or len(text_b) < 20:
                    continue
                positive, negative = (text_a, text_b) if closer else (text_b, text_a)
                self.data.append((anchor, positive, negative))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


class TrackARowsDataset(Dataset):
    def __init__(self, rows):
        self.rows_eval = list(rows)
        self.data = [
            (
                obj["anchor_text"],
                obj["text_a"],
                obj["text_b"],
                bool(obj["text_a_is_closer"]),
            )
            for obj in rows
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


class TrackATestDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path, encoding="utf-8") as handle:
            for line in handle:
                if line.strip():
                    obj = json.loads(line)
                    self.data.append((obj["anchor_text"], obj["text_a"], obj["text_b"]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


# ===================== RUNNER =====================
class ComparisonRunner:
    def __init__(self, language, paths):
        set_seed(SEED)
        cleanup_cuda()
        self.language = language
        self.paths = paths
        self.lang_label = paths["language"]
        self.safe_language = "english" if language == "en" else "romanian"
        self.run_prefix = f"{RUN_PREFIX}_{language}"
        self.dl_generator = torch.Generator()
        self.dl_generator.manual_seed(SEED)

        print("\n" + "=" * 70)
        print(f"Starting {self.lang_label} comparison run")
        print("=" * 70)
        print(f"Model: {MODEL_NAME}")
        print(f"Data dir: {paths['data_dir']}")
        print(f"Max length: {MAX_LEN}")
        print(f"Latent heads: {HEAD_KEYS}")
        print("Input: full text only")

        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=TRUST_REMOTE_CODE)
        self.encoder = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=TRUST_REMOTE_CODE).to(DEVICE)
        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False
        if DEVICE == "cuda":
            self.encoder.gradient_checkpointing_enable()
            print("Gradient checkpointing: ENABLED")

        self.hidden_dim = self.encoder.config.hidden_size
        self.model = MultilingualG2Model(self.encoder, self.hidden_dim, PROJ_DIM, HEAD_KEYS).to(DEVICE)

        self.out_track_a = OUT_DIR / f"{self.run_prefix}_track_a_predictions.jsonl"
        self.out_track_b = OUT_DIR / f"{self.run_prefix}_track_b.npy"

    def close(self):
        del self.model
        del self.encoder
        del self.tokenizer
        cleanup_cuda()

    def tokenize(self, texts):
        prepared = [MODEL_PREFIX + str(text) for text in texts]
        return self.tokenizer(
            prepared,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        ).to(DEVICE)

    def pack_forward(self, texts):
        enc = self.tokenize(texts)
        with autocast_context():
            return self.model(enc["input_ids"], enc["attention_mask"])

    def triplet_loss(self, anchor, positive, negative):
        return F.triplet_margin_loss(anchor, positive, negative, margin=MARGIN)

    def rank_loss(self, anchor, positive, negative):
        return F.relu(
            F.cosine_similarity(anchor, negative).mean()
            - F.cosine_similarity(anchor, positive).mean()
            + 0.2
        )

    def head_triplet_loss(self, out_anchor, out_positive, out_negative):
        if not HEAD_KEYS:
            return torch.tensor(0.0, device=DEVICE)
        return sum(
            self.triplet_loss(out_anchor[key], out_positive[key], out_negative[key])
            for key in HEAD_KEYS
        ) / float(len(HEAD_KEYS))

    def train_stage1(self, dataset, epochs):
        if len(dataset) == 0 or epochs <= 0:
            print("Stage1 skipped.")
            return

        loader = DataLoader(
            dataset,
            batch_size=STAGE1_BS,
            shuffle=True,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=self.dl_generator,
        )
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=STAGE1_LR)
        scaler = make_grad_scaler()
        self.model.train()

        for epoch in range(epochs):
            loop = tqdm(loader, desc=f"{self.language} Stage1 Epoch {epoch + 1}")
            running = 0.0
            for anchors, positives, negatives in loop:
                texts = list(anchors) + list(positives) + list(negatives)
                out = self.pack_forward(texts)
                batch_size = len(anchors)

                anchor_out = {key: value[:batch_size] for key, value in out.items()}
                positive_out = {key: value[batch_size:2 * batch_size] for key, value in out.items()}
                negative_out = {key: value[2 * batch_size:] for key, value in out.items()}

                global_loss = self.triplet_loss(
                    anchor_out["global"],
                    positive_out["global"],
                    negative_out["global"],
                )
                rank = self.rank_loss(
                    anchor_out["global"],
                    positive_out["global"],
                    negative_out["global"],
                )
                head_loss = self.head_triplet_loss(anchor_out, positive_out, negative_out)
                loss = GLOBAL_TRIPLET_WEIGHT * global_loss + GLOBAL_RANK_WEIGHT * rank
                loss = loss + HEAD_TRIPLET_WEIGHT * head_loss

                backward_step(loss, optimizer, scaler, self.model)
                running += float(loss.item())
                loop.set_postfix(loss=f"{running / (loop.n + 1):.4f}")

        self.model.eval()
        cleanup_cuda()

    def evaluate_rows(self, rows):
        dataset = TrackARowsDataset(rows)
        loader = DataLoader(
            dataset,
            batch_size=PRED_BS,
            shuffle=False,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
        )
        y_true, y_pred = [], []
        self.model.eval()
        with torch.no_grad():
            for anchors, text_a_list, text_b_list, labels in loader:
                texts = list(anchors) + list(text_a_list) + list(text_b_list)
                out = self.pack_forward(texts)
                batch_size = len(anchors)
                global_emb = out["global"]
                anchor = global_emb[:batch_size]
                text_a = global_emb[batch_size:2 * batch_size]
                text_b = global_emb[2 * batch_size:]
                pred = (F.cosine_similarity(anchor, text_a) > F.cosine_similarity(anchor, text_b)).cpu().tolist()
                y_pred.extend(bool(x) for x in pred)
                y_true.extend(bool(x) for x in labels)

        acc = accuracy_score(y_true, y_pred)
        macro_f1, tp, tn, fp, fn = macro_f1_from_binary(y_true, y_pred)
        return {
            "accuracy": float(acc),
            "macro_f1": float(macro_f1),
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "n": len(y_true),
        }

    def train_stage2(self, rows, epochs):
        dataset = TrackARowsDataset(rows)
        loader = DataLoader(
            dataset,
            batch_size=STAGE2_BS,
            shuffle=True,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=self.dl_generator,
        )
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=STAGE2_LR)
        scaler = make_grad_scaler()
        criterion = nn.BCEWithLogitsLoss()
        self.model.train()

        for epoch in range(epochs):
            loop = tqdm(loader, desc=f"{self.language} Stage2 Epoch {epoch + 1}")
            running = 0.0
            for anchors, text_a_list, text_b_list, labels in loop:
                y = torch.as_tensor(labels, dtype=torch.float32, device=DEVICE)
                texts = list(anchors) + list(text_a_list) + list(text_b_list)
                out = self.pack_forward(texts)
                batch_size = len(anchors)
                global_emb = out["global"]
                anchor = global_emb[:batch_size]
                text_a = global_emb[batch_size:2 * batch_size]
                text_b = global_emb[2 * batch_size:]
                score = F.cosine_similarity(anchor, text_a) - F.cosine_similarity(anchor, text_b)
                loss = criterion(score, y)

                backward_step(loss, optimizer, scaler, self.model)
                running += float(loss.item())
                loop.set_postfix(loss=f"{running / (loop.n + 1):.4f}")

            if EVAL_EACH_EPOCH:
                epoch_eval = self.evaluate_rows(dataset.rows_eval)
                print(
                    f"{self.language} Stage2 Epoch {epoch + 1} summary | "
                    f"loss={running / max(len(loader), 1):.4f} | "
                    f"acc={epoch_eval['accuracy'] * 100:.2f}% | "
                    f"macro_f1={epoch_eval['macro_f1'] * 100:.2f}%"
                )
            cleanup_cuda()

        self.model.eval()
        cleanup_cuda()

    def write_track_a_predictions(self):
        dataset = TrackATestDataset(self.paths["test_track_a"])
        loader = DataLoader(dataset, batch_size=PRED_BS, shuffle=False, num_workers=DL_NUM_WORKERS)
        preds = []
        self.model.eval()
        with torch.no_grad():
            for anchors, text_a_list, text_b_list in loader:
                texts = list(anchors) + list(text_a_list) + list(text_b_list)
                out = self.pack_forward(texts)
                batch_size = len(anchors)
                global_emb = out["global"]
                anchor = global_emb[:batch_size]
                text_a = global_emb[batch_size:2 * batch_size]
                text_b = global_emb[2 * batch_size:]
                batch_preds = (
                    F.cosine_similarity(anchor, text_a)
                    > F.cosine_similarity(anchor, text_b)
                ).cpu().tolist()
                preds.extend(batch_preds)

        with open(self.paths["test_track_a"], encoding="utf-8") as fin, open(self.out_track_a, "w", encoding="utf-8") as fout:
            for line, pred in zip(fin, preds):
                obj = json.loads(line)
                obj["text_a_is_closer"] = bool(pred)
                fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

        print(f"Saved Track A predictions: {self.out_track_a}")

    def build_track_b_embeddings(self):
        texts = []
        with open(self.paths["test_track_b"], encoding="utf-8") as handle:
            for line in handle:
                if line.strip():
                    obj = json.loads(line)
                    texts.append(obj["text"])

        embeddings = []
        self.model.eval()
        with torch.no_grad():
            for start in tqdm(range(0, len(texts), ENC_BS), desc=f"{self.language} Track B encoding"):
                batch = texts[start:start + ENC_BS]
                out = self.pack_forward(batch)
                emb = torch.cat(
                    [out["global"]] + [out[key] for key in HEAD_KEYS],
                    dim=1,
                )
                emb = F.normalize(emb, p=2, dim=1)
                embeddings.append(emb.cpu().numpy().astype(np.float32))

        x = np.vstack(embeddings).astype(np.float32)
        x = l2_normalize_np(x)
        np.save(self.out_track_b, x)
        print(f"Saved Track B embeddings: {self.out_track_b} shape={x.shape}")
        cleanup_cuda()
        return x

    def evaluate_track_a(self):
        pred = pd.read_json(self.out_track_a, lines=True)
        gold = pd.read_json(self.paths["test_track_a_labels"], lines=True)
        y_pred = pred["text_a_is_closer"].astype(bool).tolist()
        y_true = gold["text_a_is_closer"].astype(bool).tolist()
        acc = accuracy_score(y_true, y_pred)
        macro_f1, tp, tn, fp, fn = macro_f1_from_binary(y_true, y_pred)
        return {
            "accuracy": float(acc),
            "macro_f1": float(macro_f1),
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "n": len(y_true),
        }

    def evaluate_track_b(self):
        pred = pd.read_json(self.paths["test_track_b"], lines=True)
        embeddings = np.load(self.out_track_b, allow_pickle=False)
        embeddings = l2_normalize_np(embeddings.astype(np.float32))

        if len(pred) != embeddings.shape[0]:
            raise ValueError(f"Track B input rows={len(pred)} but embeddings rows={embeddings.shape[0]}")

        lookup = dict(zip(pred["text"].astype(str).map(norm_text), embeddings))
        gold = pd.read_json(self.paths["test_track_b_labels"], lines=True)

        missing_translation_fields = 0
        for col in ["anchor_text", "text_a", "text_b"]:
            values = []
            for value in gold[col].astype(str).tolist():
                if self.language == "ro":
                    translated = translate_ref_text_for_eval(value)
                    if translated is None:
                        missing_translation_fields += 1
                        translated = norm_text(value)
                    values.append(translated)
                else:
                    values.append(norm_text(value))
            gold[col] = values

        gold["anchor_embedding"] = gold["anchor_text"].map(lookup)
        gold["a_embedding"] = gold["text_a"].map(lookup)
        gold["b_embedding"] = gold["text_b"].map(lookup)
        missing_embedding_rows = int(
            (gold["anchor_embedding"].isna() | gold["a_embedding"].isna() | gold["b_embedding"].isna()).sum()
        )
        if missing_embedding_rows:
            gold = gold[
                ~(gold["anchor_embedding"].isna() | gold["a_embedding"].isna() | gold["b_embedding"].isna())
            ].copy()

        y_true = gold["text_a_is_closer"].astype(bool).tolist()
        y_pred = [
            float(row["anchor_embedding"].dot(row["a_embedding"]))
            > float(row["anchor_embedding"].dot(row["b_embedding"]))
            for _, row in gold.iterrows()
        ]
        acc = accuracy_score(y_true, y_pred)
        macro_f1, tp, tn, fp, fn = macro_f1_from_binary(y_true, y_pred)
        return {
            "accuracy": float(acc),
            "macro_f1": float(macro_f1),
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "n": len(y_true),
            "missing_translation_fields": int(missing_translation_fields),
            "missing_embedding_rows": int(missing_embedding_rows),
            "embedding_dim": int(embeddings.shape[1]),
        }

    def run(self):
        for key in ["dev_track_a", "test_track_a", "test_track_b"]:
            if not Path(self.paths[key]).exists():
                raise FileNotFoundError(f"Missing {self.language} {key}: {self.paths[key]}")

        if USE_SYNTHETIC_STAGE1:
            datasets = [SyntheticTripletDataset(self.paths["synth"])]
            if USE_EXTRA_SYNTHETIC:
                datasets.append(SyntheticTripletDataset(self.paths["extra"]))
            synth_dataset = ConcatDataset(datasets)
            print(f"{self.language} Stage1 synthetic triples: {len(synth_dataset)}")
            self.train_stage1(synth_dataset, STAGE1_EPOCHS)
            
        torch.cuda.empty_cache() 
        if not RUN_FINAL_TEST:
            return {"language": self.lang_label, "stage1_only": True}

        dev_rows = load_jsonl(self.paths["dev_track_a"])
        print(f"{self.language} dev rows: {len(dev_rows)}")
        self.train_stage2(dev_rows, STAGE2_EPOCHS)
        self.write_track_a_predictions()
        self.build_track_b_embeddings()
        res_a = self.evaluate_track_a()
        res_b = self.evaluate_track_b()

        result = {
            "language_code": self.language,
            "language": self.lang_label,
            "paths": {key: str(value) for key, value in self.paths.items()},
            "outputs": {
                "track_a_predictions": str(self.out_track_a),
                "track_b_embeddings": str(self.out_track_b),
            },
            "track_a": res_a,
            "track_b": res_b,
        }

        print("\n" + "-" * 60)
        print(f"{self.lang_label} summary")
        print(f"Track A accuracy : {res_a['accuracy'] * 100:.2f}%")
        print(f"Track A macro-F1 : {res_a['macro_f1'] * 100:.2f}%")
        print(f"Track B accuracy : {res_b['accuracy'] * 100:.2f}%")
        print(f"Track B macro-F1 : {res_b['macro_f1'] * 100:.2f}%")
        print("-" * 60)

        return result


def write_markdown_summary(payload):
    lines = []
    lines.append("# Multilingual G2 English vs Romanian Comparison")
    lines.append("")
    lines.append(f"- Model: `{payload['model_name']}`")
    lines.append(f"- Max length: `{payload['max_len']}`")
    lines.append(f"- Heads: `{', '.join(payload['head_keys'])}`")
    lines.append(f"- Seed: `{payload['seed']}`")
    lines.append("- Input: `full text only`")
    lines.append("")
    lines.append("| Language | Track A Acc. | Track A Macro-F1 | Track B Acc. | Track B Macro-F1 |")
    lines.append("|---|---:|---:|---:|---:|")
    for code in payload["language_order"]:
        result = payload["results"][code]
        lines.append(
            f"| {result['language']} | "
            f"{result['track_a']['accuracy'] * 100:.2f} | "
            f"{result['track_a']['macro_f1'] * 100:.2f} | "
            f"{result['track_b']['accuracy'] * 100:.2f} | "
            f"{result['track_b']['macro_f1'] * 100:.2f} |"
        )
    if "deltas" in payload:
        lines.append("")
        lines.append("## Deltas")
        lines.append("")
        for key, value in payload["deltas"].items():
            lines.append(f"- {key}: {value * 100:.2f} percentage points")
    COMPARISON_MD.write_text("\n".join(lines) + "\n", encoding="utf-8")


def main():
    print("\n" + "=" * 70)
    print("MULTILINGUAL G2-STYLE ENGLISH/ROMANIAN COMPARISON")
    print("=" * 70)
    print(f"Languages: {LANGUAGES}")
    print(f"Model: {MODEL_NAME}")
    print(f"Model prefix: {MODEL_PREFIX!r}")
    print(f"Use AMP: {USE_AMP}")
    print("Input: full text only")
    print(f"Output prefix: {RUN_PREFIX}")

    results = {}
    for language in LANGUAGES:
        paths = language_paths(language)
        runner = ComparisonRunner(language, paths)
        try:
            results[language] = runner.run()
        finally:
            runner.close()

    payload = {
        "run_prefix": RUN_PREFIX,
        "seed": SEED,
        "model_name": MODEL_NAME,
        "model_prefix": MODEL_PREFIX,
        "trust_remote_code": TRUST_REMOTE_CODE,
        "max_len": MAX_LEN,
        "head_keys": HEAD_KEYS,
        "proj_dim": PROJ_DIM,
        "stage1_epochs": STAGE1_EPOCHS,
        "stage2_epochs": STAGE2_EPOCHS,
        "stage1_lr": STAGE1_LR,
        "stage2_lr": STAGE2_LR,
        "stage1_bs": STAGE1_BS,
        "stage2_bs": STAGE2_BS,
        "pred_bs": PRED_BS,
        "enc_bs": ENC_BS,
        "use_synthetic_stage1": USE_SYNTHETIC_STAGE1,
        "use_extra_synthetic": USE_EXTRA_SYNTHETIC,
        "input_mode": "full_text",
        "head_triplet_weight": HEAD_TRIPLET_WEIGHT,
        "global_triplet_weight": GLOBAL_TRIPLET_WEIGHT,
        "global_rank_weight": GLOBAL_RANK_WEIGHT,
        "use_amp": USE_AMP,
        "en_data_dir": str(EN_DATA_DIR),
        "ro_data_dir": str(RO_DATA_DIR),
        "translation_cache_path": str(TRANSLATION_CACHE_PATH),
        "translation_cache_entries": len(TRANSLATION_CACHE),
        "language_order": LANGUAGES,
        "results": results,
    }

    if "en" in results and "ro" in results and RUN_FINAL_TEST:
        payload["deltas"] = {
            "track_a_ro_minus_en_accuracy": results["ro"]["track_a"]["accuracy"] - results["en"]["track_a"]["accuracy"],
            "track_a_ro_minus_en_macro_f1": results["ro"]["track_a"]["macro_f1"] - results["en"]["track_a"]["macro_f1"],
            "track_b_ro_minus_en_accuracy": results["ro"]["track_b"]["accuracy"] - results["en"]["track_b"]["accuracy"],
            "track_b_ro_minus_en_macro_f1": results["ro"]["track_b"]["macro_f1"] - results["en"]["track_b"]["macro_f1"],
        }

    COMPARISON_JSON.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    write_markdown_summary(payload)

    print("\n" + "=" * 70)
    print("COMPARISON SUMMARY")
    print("=" * 70)
    for language in LANGUAGES:
        result = results[language]
        if "track_a" not in result:
            continue
        print(
            f"{result['language']}: "
            f"Track A={result['track_a']['accuracy'] * 100:.2f}% | "
            f"Track B={result['track_b']['accuracy'] * 100:.2f}%"
        )
    print(f"Saved metrics: {COMPARISON_JSON}")
    print(f"Saved markdown summary: {COMPARISON_MD}")


if __name__ == "__main__":
    main()

Languages: ['en', 'ro']
# **Model: intfloat/multilingual-e5-base**
Max length: 512, Latent heads: ['head1', 'head2'], Input: full text only


**English summary**

    Track A accuracy : 70.00%
    Track A macro-F1 : 69.95%
    Track B accuracy : 68.25%
    Track B macro-F1 : 68.23%

**Romanian machine translation summary**

    Track A accuracy : 66.25%
    Track A macro-F1 : 66.17%
    Track B accuracy : 63.50%
    Track B macro-F1 : 63.44%

**COMPARISON SUMMARY**

    English: Track A=70.00% | Track B=68.25%
    Romanian machine translation: Track A=66.25% | Track B=63.50%

# **Model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2**

Max length: 512, Latent heads: ['head1', 'head2'], Input: full text only

**English summary**

    Track A accuracy : 65.00%
    Track A macro-F1 : 65.00%
    Track B accuracy : 63.00%
    Track B macro-F1 : 62.94%


**Romanian machine translation summary**

    Track A accuracy : 62.00%
    Track A macro-F1 : 62.00%
    Track B accuracy : 59.00%
    Track B macro-F1 : 58.99%


**COMPARISON SUMMARY**

    English: Track A=65.00% | Track B=63.00%
    Romanian machine translation: Track A=62.00% | Track B=59.00%
